# A2 · reproduce the expert job-shop results (JaamSim, Zenodo 23229885)
Runs the expert M3 model headless and prints our numbers next to the published ones.
**Stages:** 1 setup + download (checks MD5) · 2 inspect the cfg and the run output · 3 run + compare.
Expert values: `expert_values.json` (M3 norm 3.31 h, over20 11.54 h; 4-worker variant = our own re-run, not published).

In [ ]:
# 1a. Get the repo
import os, subprocess
if not os.path.isdir('planner-copilot'):
    subprocess.run(['git', 'clone', 'https://github.com/bodhi584/planner-copilot'], check=True)
%cd planner-copilot/cases/A2_sme

In [ ]:
# 1b. Build JaamSim headless (a few minutes, once)
!bash setup.sh

In [ ]:
# 1c. Download the model files from Zenodo and verify MD5 (values from the Zenodo record page)
import hashlib, os, urllib.request
BASE = 'https://zenodo.org/records/23229885/files/'
MD5 = {
 'M1_weekly_backlog.cfg':'997a4fcf1fc123eafe8fbeecb64353d8',
 'M2_reference_for_M3.cfg':'af62baa3183b4307e6bb9956f68a6c76',
 'M3_weekly_backlog.cfg':'f4fc73696ac4ec9cba944ba4b67e0fde',
 'portf_norm_clean.tsv':'8196e508e9998827b6a4f2c756fb795b',
 'portf_over20_clean.tsv':'3e8caad85102e228bf1771b890919024',
 'M3_norm_30replikacji.rep':'4eec163221cf56ed643ca19f020117f3',
 'M3_over20_30replikacji.rep':'eccbe8e883b993874a5903b6f35cffa3',
}
os.makedirs('model', exist_ok=True)
for name, md5 in MD5.items():
    path = os.path.join('model', name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + name + '?download=1', path)
    got = hashlib.md5(open(path,'rb').read()).hexdigest()
    print(('OK  ' if got == md5 else 'BAD '), name, got)

In [ ]:
# 2a. Inspect the M3 cfg: where are workers, shift length, breaks, replications, report settings?
!grep -n -i -E "worker|employee|resource|shift|break|replication|seed|report|duration|runtime" model/M3_weekly_backlog.cfg | head -80

In [ ]:
# 2b. Run both published scenarios (3 workers) headless
!bash run.sh M3 norm
!bash run.sh M3 over20

In [ ]:
# 2c. Look at what the run produced (check the file layout before parsing)
!ls -la runs/M3_norm runs/M3_over20
!head -60 runs/M3_norm/m.rep
!head -15 runs/M3_norm/m-OrderResults.log
!head -60 model/M3_norm_30replikacji.rep

## 3. Compare with the published values
Read straight from the run files (`runs/M3_*/m.rep`) and from the authors' 30-replication files (`model/M3_*_30replikacji.rep`).
* overtime after Friday shift end (h) = `CompleteOrder PortfolioCompletion` (h) minus 104 h (Friday shift end = minute 6240 in `ShiftCalendar`)
* unfinished orders = `FridaySnapshot UnfinishedOrders`; utilisation = `<machine> Utilisation`
* pass rule (acceptance standard): our mean lies within published mean +- 2 sd, 30 replications. We also report the largest per-replication difference.

In [ ]:
import json, statistics as st
TAB = chr(9)
def parse(path):
    reps, cur = [], None
    for line in open(path, encoding='latin-1'):
        if line.startswith('##### SCENARIO'):
            cur = {}; reps.append(cur); continue
        p = line.rstrip().split(TAB)
        if cur is not None and len(p) >= 4:
            cur[(p[0], p[1])] = (p[2], p[3])
    return reps
FRIDAY_END_H = 104.0   # ShiftCalendar: Friday shift ends at minute 6240
def kpi(reps):
    rows = []
    for r in reps:
        pc = r[('CompleteOrder', 'PortfolioCompletion')]
        assert pc[1] == 'h', pc
        rows.append(dict(ot=float(pc[0]) - FRIDAY_END_H,
                         unf=float(r[('FridaySnapshot', 'UnfinishedOrders')][0]),
                         sheet=float(r[('SheetLaser', 'Utilisation')][0]), tube=float(r[('TubeLaser', 'Utilisation')][0]),
                         punch=float(r[('PunchingMachine', 'Utilisation')][0]), bend=float(r[('BendingMachine', 'Utilisation')][0])))
    return rows
EXP = json.load(open('expert_values.json'))
results = {}
for P in ['norm', 'over20']:
    ours = kpi(parse(f'runs/M3_{P}/m.rep')); pub = kpi(parse(f'model/M3_{P}_30replikacji.rep'))
    results[P] = {'n_ours': len(ours), 'n_published': len(pub)}
    for k in ['ot', 'unf', 'sheet', 'tube', 'punch', 'bend']:
        a = [r[k] for r in ours]; b = [r[k] for r in pub]
        results[P][k] = dict(ours_mean=st.mean(a), pub_mean=st.mean(b), pub_sd=st.stdev(b), max_abs_diff_per_rep=max(abs(x - y) for x, y in zip(a, b)))
    o = results[P]['ot']
    results[P]['pass'] = abs(o['ours_mean'] - o['pub_mean']) <= 2 * o['pub_sd'] and len(ours) >= 30
    print(P, 'overtime h: ours %.4f | published %.4f (sd %.4f) | expert_values.json %.2f | max per-rep diff %.1e | PASS=%s' % (
        o['ours_mean'], o['pub_mean'], o['pub_sd'], EXP[f'M3_{P}_3workers']['overtime_h'], o['max_abs_diff_per_rep'], results[P]['pass']))
    u = results[P]['unf']
    print('   unfinished orders: ours %.4f | published %.4f (sd %.4f) | expert_values.json %.1f' % (u['ours_mean'], u['pub_mean'], u['pub_sd'], EXP[f'M3_{P}_3workers']['unfinished_orders']))
    print('   utilisation (share of simulated time):', {k: round(results[P][k]['ours_mean'], 4) for k in ['sheet', 'tube', 'punch', 'bend']})
json.dump(results, open('reproduce_results.json', 'w'), indent=1)

In [ ]:
# 4-worker variant = OUR reference run, not a published scenario.
# The cfg defines Worker1-3 (+ Worker1-3Break, 20 min inside a 180-240 min window, staggered 0 / 0.5 / 1 of the window).
# How a 4th worker's break is staggered is not given anywhere: UNVERIFIED assumption, so we run two variants.
import os, re, subprocess
src = open('model/M3_weekly_backlog.cfg', encoding='latin-1').read().splitlines()
def make(fr, P):
    res = []
    for l in src:
        if l.startswith('Define DowntimeEntity { Worker1Break'): l = l.replace('Worker3Break }', 'Worker3Break  Worker4Break }')
        if l.startswith('Define ResourceUnit { Worker1'): l = l.replace('Worker3 }', 'Worker3  Worker4 }')
        m = re.match(r'Worker([123])Break FirstDowntime', l)
        if m: l = "Worker%dBreak FirstDowntime { '[BreakWindowStart].Value + %s * ([BreakWindowEnd].Value - [BreakWindowStart].Value)' }" % (int(m.group(1)), fr[int(m.group(1)) - 1])
        if l.startswith('Orders DataFile'): l = 'Orders DataFile { portf_%s_clean.tsv }' % P
        res.append(l)
    add = ["Worker4Break FirstDowntime { '[BreakWindowStart].Value + %s * ([BreakWindowEnd].Value - [BreakWindowStart].Value)' }" % fr[3]]
    add += [l.replace('Worker3', 'Worker4') for l in src if re.match(r'Worker3Break (Interval|Duration|MaxDowntimesPending) ', l) or re.match(r'Worker3 (ForcedMaintenanceList|ResourcePool) ', l)]
    return res + add
J = os.path.abspath('jaamsim')
CP = ':'.join([J + '/classes', J + '/src/main/resources', J + '/jar/jogl-all.jar', J + '/jar/gluegen-rt.jar'])
variants = {'even': (0, 0.333333, 0.666667, 1), 'quarter': (0, 0.5, 1, 0.25)}
four_res = {}
for name, fr in variants.items():
    for P in ['norm', 'over20']:
        D = f'runs/W4_{name}_{P}'; os.makedirs(D, exist_ok=True)
        open(D + '/m.cfg', 'w', encoding='latin-1').write(chr(10).join(make(fr, P)) + chr(10))
        for f in ['portf_norm_clean.tsv', 'portf_over20_clean.tsv']: open(D + '/' + f, 'wb').write(open('model/' + f, 'rb').read())
        subprocess.run(['java', '-Djava.awt.headless=true', '-cp', CP, 'com.jaamsim.ui.GUIFrame', 'm.cfg', '-b', '-h'], cwd=D, capture_output=True, text=True)
        rows = kpi(parse(D + '/m.rep'))
        ot = [r['ot'] for r in rows]
        four_res[f'{name}_{P}'] = dict(n=len(ot), ot_mean=st.mean(ot), ot_sd=st.stdev(ot))
        print(f'{name:8s} {P:6s} n={len(ot)} overtime mean {st.mean(ot):.3f} sd {st.stdev(ot):.3f} | pilot reference {EXP["M3_" + P + "_4workers"]["overtime_h"]}')
results['four_workers'] = four_res
json.dump(results, open('reproduce_results.json', 'w'), indent=1)